# Loading and processing workload and index recommendations from Yanex artifacts

In [ ]:
WORKLOAD_EXP_ID = "<workload-experiment-id>"
RECOMMEND_EXP_ID = "<recommendation-experiment-id>"

## Query patterns

In [ ]:
import yanex.results as yr
from bondi.mongodb import MongoQueryPattern

experiment = yr.get_experiment(WORKLOAD_EXP_ID)
summaries = experiment.load_artifact("pattern_summary.json")

# Restore each saved pattern for readable printing
for summary in summaries:
    pattern = MongoQueryPattern.from_dict(summary["pattern"])
    print(f"Pattern {summary['pattern_id']}: {pattern}")

## Queries grouped by pattern

In [ ]:
from collections import defaultdict

from bondi.workload import Workload

workload = experiment.load_artifact("workload.jsonl", loader=Workload.load)

# Group queries by their pattern ID
queries_by_pattern = defaultdict(list)
for entry in workload:
    queries_by_pattern[entry.pattern_id].append(entry.query)

# Print patterns in ID order, with unassigned queries last.
for pattern_id in sorted(queries_by_pattern, key=lambda pid: (pid is None, pid)):
    queries = queries_by_pattern[pattern_id]
    label = f"Pattern {pattern_id}" if pattern_id is not None else "Unassigned"
    print(f"\n{label} ({len(queries)} queries)")

    for i, query in enumerate(queries, start=1):
        print(
            f"  {i}. Filter: {query.filter}, "
            f"Projection: {query.projection}, Sort: {query.sort}, "
            f"Skip: {query.skip}, Limit: {query.limit}"
        )

## APA Index Recommendations

In [ ]:
import yanex.results as yr

# Load the recommendations from a completed Yanex recommendation experiment.
experiment = yr.get_experiment(RECOMMEND_EXP_ID)
recommendations = experiment.load_artifact("recommendations.json")

# Already ordered by descending APA score; preserve this order.
indexes = recommendations["indexes"]

for rank, index in enumerate(indexes, start=1):
    score = index["score"]  # APA weight
    keys = index["keys"]  # ordered pairs, e.g. [["city", 1], ["stars", -1]]

    # PyMongo expects a list of tuples for compound index keys.
    keys = [(field, direction) for field, direction in keys]

    # Use rank, keys and score in your own code.
    print(f"Rank: {rank}, Keys: {keys}, Score: {score}")